# IgniteCyber 5-Day Cyber AI Analyst Bootcamp - LAB2.5

**Voice Fraud & Voice Security Triage**  
**Offline-first** - **Defensive SOC workflow** - **Elastic/OpenSearch + Wazuh/Zeek + TheHive + MISP + Ollama**

AI output must be treated as an analyst assistant, not as evidence. Students must validate all findings against the source logs, transcript, case artifacts, and CTI.


## 1. Lab overview

Day 2 now includes voice fraud and voice security triage as part of the phishing and social engineering storyline. This lab uses synthetic call records, transcripts, help desk events, identity events, endpoint logs, Zeek-style HTTP telemetry, MISP CTI, and a TheHive case bundle.

This is a defensive lab. It does not include voice cloning, deepfake generation, authentication bypass instructions, real customer phone numbers, or real Avaya CM configuration.


## 2. Scenario brief

ApexFin Solutions receives a suspicious call to the help desk. The caller claims to be a senior executive traveling for a board meeting and urgently requests an MFA reset. The caller provides partial employee information and pressures the help desk technician to bypass normal verification. Shortly after the call, identity logs show a successful MFA device rebind attempt, followed by a suspicious login from a new device and browsing activity to a suspicious domain.

- Organization: ApexFin Solutions
- Threat actor: Cinder Jackal
- Target role: Help Desk Tier 1
- Claimed identity: ApexFin CFO
- Victim user: `mrivera@apexfin.example`
- Suspicious caller ID: `+1-202-555-0148`
- Callback number: `+1-202-555-0199`
- Suspicious domain: `apexfin-helpdesk-login.example`
- Case ID: `CJ-VOICE-001`


## 3. Learning objectives

- Identify social engineering indicators in call metadata and transcripts.
- Correlate help desk, identity, endpoint, and network telemetry by `case_id`, `ticket_id`, `user`, `source_ip`, and `device_id`.
- Enrich suspicious voice fraud observables in MISP and manage response tasks in TheHive.
- Draft detection logic for voice-driven MFA reset abuse.
- Use AI summarization defensively while validating all model output against evidence.


## 4. Dataset loading

The lab uses local synthetic JSON and transcript files from `datasets/LAB-2.5`. If running inside the standard bootcamp container, adjust `DATASET_DIR` if the pack is mounted somewhere else.


In [ ]:
import json
from pathlib import Path
from collections import Counter

try:
    from bootcamp_lib import ollama_stream, ollama_models
except Exception:
    ollama_stream = None
    def ollama_models():
        return []

DATASET_DIR = Path('../datasets/LAB-2.5')
if not DATASET_DIR.exists():
    DATASET_DIR = Path('/opt/bootcamp/datasets/LAB-2.5')

def load_json(name):
    with open(DATASET_DIR / name, 'r', encoding='utf-8') as f:
        return json.load(f)

call_records = load_json('voice_call_records.json')
helpdesk_events = load_json('helpdesk_ticket_events.json')
mfa_events = load_json('mfa_identity_events.json')
endpoint_events = load_json('voice_fraud_endpoint_events.json')
zeek_http = load_json('voice_fraud_zeek_http.json')
transcripts = sorted((DATASET_DIR / 'voice_fraud_transcripts').glob('*.txt'))

print('Dataset directory:', DATASET_DIR)
for name, rows in [('call_records', call_records), ('helpdesk_events', helpdesk_events), ('mfa_events', mfa_events), ('endpoint_events', endpoint_events), ('zeek_http', zeek_http)]:
    print(f'{name}: {len(rows)} rows')
print('transcripts:', [p.name for p in transcripts])


## 5. Call-detail record review

Start with the call-detail records. Look for urgency, executive impersonation, attestation issues, mismatched callback numbers, and MFA reset pressure.


In [ ]:
for row in call_records:
    flags = ', '.join(row.get('risk_flags', [])) or 'none'
    print(f"{row['timestamp']} | {row['case_id']} | caller={row['caller_id']} | callback={row['callback_number']} | attestation={row['stir_shaken_attestation']} | result={row['call_result']} | flags={flags}")

flag_counts = Counter(flag for row in call_records for flag in row.get('risk_flags', []))
print('\nRisk flag counts:', dict(flag_counts))


## 6. Transcript review

Review transcripts for social engineering markers: urgency, authority pressure, refusal to use standard verification, alternate callback number, MFA reset/device rebind request, and attempts to move outside normal process.


In [ ]:
for path in transcripts:
    print(f'\n--- {path.name} ---')
    text = path.read_text(encoding='utf-8')
    print(text[:1200])


## 7. Help desk ticket analysis

Use `ticket_id`, `case_id`, `caller_id`, and `user` to connect the call with help desk workflow decisions.


In [ ]:
for event in helpdesk_events:
    print(f"{event['timestamp']} | {event['ticket_id']} | {event['event_type']} | {event['user']} | {event['details']}")


## 8. MFA / identity event correlation

Correlate the reset request, device rebind, new device login, conditional access challenge, and risk-score increase.


In [ ]:
for event in mfa_events:
    print(f"{event['timestamp']} | {event['event_type']} | user={event['user']} | ip={event['source_ip']} | device={event['device_id']} | result={event['result']} | risk={event['risk_score']} | {event['details']}")


## 9. Endpoint and network activity correlation

Look for suspicious URL access, download behavior, script execution, and HTTP activity that occurs after the voice contact and MFA rebind.


In [ ]:
timeline = []
for group_name, rows, ts_key in [
    ('call', call_records, 'timestamp'),
    ('helpdesk', helpdesk_events, 'timestamp'),
    ('identity', mfa_events, 'timestamp'),
    ('endpoint', endpoint_events, '@timestamp'),
    ('zeek_http', zeek_http, 'ts'),
]:
    for row in rows:
        if row.get('case_id') == 'CJ-VOICE-001':
            timeline.append((row[ts_key], group_name, row))

for ts, group, row in sorted(timeline, key=lambda x: x[0]):
    label = row.get('event_type') or row.get('event.action') or row.get('call_result') or row.get('uri')
    print(f'{ts} | {group:9} | {label}')


## 10. MISP enrichment

Open `labs/LAB-2.5/misp/LAB-2_5_voice_fraud_misp_event.json`. Confirm that the suspicious caller ID, callback number, domain, URL, source IP, threat actor, and ATT&CK mappings are represented as CTI attributes.


In [ ]:
misp_path = Path('../labs/LAB-2.5/misp/LAB-2_5_voice_fraud_misp_event.json')
if misp_path.exists():
    misp = json.loads(misp_path.read_text())
    attrs = misp['Event']['Attribute']
    for attr in attrs:
        print(f"{attr['type']:8} | {attr['value']} | {attr.get('comment','')}")
else:
    print('MISP event file not found from this notebook path; inspect it manually in labs/LAB-2.5/misp/.')


## 11. TheHive case update

Open `labs/LAB-2.5/thehive/casebundle_lab2_5_voice_fraud.json`. Use the tasks to track review, enrichment, detection drafting, and final reporting.


In [ ]:
case_path = Path('../labs/LAB-2.5/thehive/casebundle_lab2_5_voice_fraud.json')
if case_path.exists():
    case = json.loads(case_path.read_text())
    print(case['case']['title'])
    print('Tasks:')
    for task in case['tasks']:
        print('-', task['title'])
    print('Observables:', len(case['observables']))
else:
    print('TheHive case bundle not found from this notebook path; inspect it manually in labs/LAB-2.5/thehive/.')


## 12. Detection logic

Detection concept: a suspicious inbound help desk call plus MFA reset/device rebind pressure, failed or skipped verification, callback mismatch, and a new device/risky login within 60 minutes should generate a high-severity voice fraud alert.

Review `labs/LAB-2.5/detections/sigma_voice_fraud_helpdesk_mfa_reset.yml` and `labs/LAB-2.5/detections/opensearch_voice_fraud_queries.md`.


In [ ]:
suspicious_cases = set()
for hd in helpdesk_events:
    if hd['event_type'] in {'verification_partial', 'callback_mismatch', 'mfa_reset_requested'}:
        suspicious_cases.add(hd['case_id'])
for ident in mfa_events:
    if ident['case_id'] in suspicious_cases and ident['event_type'] in {'mfa_device_rebound', 'new_device_login', 'identity_risk_increase'} and ident['risk_score'] >= 80:
        print('Candidate alert:', ident['case_id'], ident['user'], ident['event_type'], ident['source_ip'], ident['risk_score'])


## 13. AI-assisted triage with Ollama/SecAssist

Use local Ollama if available to summarize the transcript, extract social engineering indicators, identify suspicious entities, suggest ATT&CK mappings, and draft a TheHive case summary.

**Warning:** AI output must be treated as an analyst assistant, not as evidence. Students must validate all findings against the source logs, transcript, case artifacts, and CTI.


In [ ]:
transcript = (DATASET_DIR / 'voice_fraud_transcripts' / 'call_001_transcript.txt').read_text(encoding='utf-8')
prompt = f'''You are a SOC analyst assistant. Use only the transcript below.
Return: summary, social engineering indicators, suspicious entities, ATT&CK mapping suggestions, TheHive case summary, executive summary, and validation gaps.
Do not invent evidence.

TRANSCRIPT:
{transcript}
'''

models = ollama_models()
print('Available Ollama models:', models[:5])
if ollama_stream and models:
    ollama_stream(prompt, model=models[0])
else:
    print('Ollama helper/model not available. Read the prompt above and complete the AI-assist exercise manually or through the approved local model workflow.')


## 14. Human validation checklist

Before writing the report, verify:

- The suspicious caller ID and callback number appear in call records.
- The transcript contains urgency, authority pressure, and refusal of standard verification.
- Help desk events show partial verification, callback mismatch, and MFA reset request.
- Identity events show MFA rebind, new device login, and increased risk.
- Endpoint and Zeek events show suspicious URL access after the voice contact.
- MISP and TheHive artifacts contain only validated observables.
- No final claim relies only on AI output.


## 15. Final analyst report

Produce:

1. Short incident timeline
2. Suspicious indicators
3. MISP enrichment summary
4. TheHive case summary
5. Detection idea
6. Final incident report
7. Recommended mitigations

Recommended mitigations should include strong help desk identity verification, callback to known corporate directory number, MFA reset approval workflow, phishing-resistant MFA where available, training for executive impersonation and voice fraud, monitoring for suspicious MFA reset patterns, and a reminder that caller ID is not identity proof.


In [ ]:
report = {
    'case_id': 'CJ-VOICE-001',
    'probable_activity': 'Voice fraud / vishing attempt leading to help desk MFA reset abuse and risky new device login.',
    'key_indicators': ['+1-202-555-0148', '+1-202-555-0199', 'apexfin-helpdesk-login.example', 'https://apexfin-helpdesk-login.example/verify', '203.0.113.45'],
    'mitre': ['T1566.004', 'T1598.004', 'T1656', 'T1078'],
    'status': 'Draft - validate each item against source evidence before submission'
}
print(json.dumps(report, indent=2))


## Daily Executive Summary Checkpoint

Use this checkpoint at the end of each lab day to build the rolling executive summary for the Cinder Jackal investigation. The AI assistant can help turn validated evidence into executive-ready language, but it must not introduce new facts.

**Required workflow:**
1. Confirm the lab evidence packet or report block is filled in.
2. Ask the local AI assistant to draft a concise executive update from that evidence only.
3. Validate every claim against notebook output, Elastic/OpenSearch results, TheHive tasks, MISP updates, and source artifacts.
4. Save the final daily checkpoint under `/opt/bootcamp/reports/cj/` and append approved content to the weekly CJ Event Report.


In [ ]:
# Daily Executive Summary Checkpoint - AI-assisted, evidence-first
from datetime import datetime
import os, textwrap, json

_checkpoint_lab = globals().get("LAB_CODE", "LAB-UNKNOWN")
_checkpoint_day = globals().get("DAY_LABEL", "") or os.getenv("BOOTCAMP_DAY", "") or "Day checkpoint"
_checkpoint_ts = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%SZ")

# Prefer the validated CJ Event Report block when the notebook has one.
_checkpoint_evidence = ""
if "report_block" in globals() and str(report_block).strip():
    _checkpoint_evidence = str(report_block)
elif "findings" in globals():
    _checkpoint_evidence = json.dumps(findings, indent=2)
elif "report" in globals():
    _checkpoint_evidence = json.dumps(report, indent=2)
else:
    _checkpoint_evidence = "No structured report block found yet. Fill in the lab findings before finalizing this checkpoint."

_daily_exec_prompt = f"""
You are assisting a SOC analyst with an executive update for the IgniteCyber bootcamp.
Use only the validated evidence below. Do not invent facts, impacts, victims, or containment actions.
If evidence is missing, say what must be validated before leadership reporting.

Return this exact structure:
1. Executive summary: 3-5 bullets, business-focused, no jargon unless necessary.
2. What changed today: short paragraph.
3. Current risk: Low/Medium/High with evidence-based justification.
4. Validated evidence: 3-6 bullets with field names, artifacts, or source systems.
5. Decisions or actions needed: 2-4 bullets.
6. Open questions: 2-4 bullets.
7. Analyst validation checklist: claims that must be checked before sending.

Lab: {_checkpoint_lab}
Day: {_checkpoint_day}
Timestamp UTC: {_checkpoint_ts}

VALIDATED EVIDENCE PACKET:
{_checkpoint_evidence}
""".strip()

_daily_template = f"""# {_checkpoint_day} - Executive Summary Checkpoint

**Lab:** {_checkpoint_lab}  
**Timestamp (UTC):** {_checkpoint_ts}  
**Status:** Draft - requires analyst validation

## Executive Summary
- (AI-assisted draft or analyst-written summary goes here.)

## What Changed Today
- (Summarize the investigation progress from this lab.)

## Current Risk
- (Low/Medium/High and why.)

## Validated Evidence
- (Cite source systems, fields, observables, or artifacts.)

## Decisions or Actions Needed
- (List leadership or SOC actions.)

## Open Questions
- (List unknowns and validation gaps.)

## Analyst Validation Checklist
- [ ] Claims match source evidence.
- [ ] No AI-only claims remain.
- [ ] IOCs are validated before MISP promotion.
- [ ] TheHive tasks reflect current status.
- [ ] Executive wording is accurate and appropriately scoped.
"""

print(_daily_template)
print("\n--- AI prompt prepared for local assistant ---\n")

_models = ollama_models() if "ollama_models" in globals() else []
if _models and "ollama_stream" in globals():
    print(f"Using Ollama model: {_models[0]}\n")
    ollama_stream(_daily_exec_prompt, model=_models[0], temperature=0.2)
elif _models and "ollama_generate" in globals():
    print(f"Using Ollama model: {_models[0]}\n")
    print(ollama_generate(_daily_exec_prompt, model=_models[0], temperature=0.2))
else:
    print("No local Ollama helper/model is available in this notebook session.")
    print("Use the template above, or paste _daily_exec_prompt into the approved local AI workflow.")

_out_dir = "/opt/bootcamp/reports/cj"
os.makedirs(_out_dir, exist_ok=True)
_safe_lab = _checkpoint_lab.replace(".", "_").replace(" ", "_").replace("-", "_")
_out_path = os.path.join(_out_dir, f"{_safe_lab}_daily_executive_summary_checkpoint.md")
with open(_out_path, "w", encoding="utf-8") as f:
    f.write(_daily_template)

print(f"\nSaved daily executive summary checkpoint template: {_out_path}")


## 16. Reflection questions

- Which evidence made the event malicious rather than merely suspicious?
- What help desk control would have stopped the MFA reset?
- What telemetry was most useful: call records, ticket history, identity logs, endpoint logs, or Zeek HTTP? Why?
- What should be tuned to avoid false positives from legitimate executive travel or planned device changes?
- Where did AI help, and where could it mislead an analyst?
